# Phase 5: Independent MulCas Reconstruction

This notebook presents the frozen independent reconstruction of Algorithm 2. Published metrics are reference values from the paper; experimental metrics are generated by this repository and are labeled separately.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

results_path = PROJECT_ROOT / "results" / "baseline" / "mulcas_reproduction_results.json"
trace_dir = PROJECT_ROOT / "results" / "baseline" / "mulcas_traces"
if not results_path.exists():
    raise FileNotFoundError(
        "Run '.\\.venv\\Scripts\\python.exe -m src.models.run_mulcas' first."
    )

results = json.loads(results_path.read_text(encoding="utf-8"))
validation_trace = pd.read_csv(trace_dir / "validation_trace.csv")
test_trace = pd.read_csv(trace_dir / "test_trace.csv")
results["methodology"]

## Frozen Configuration

In [ ]:
config = results["configuration"]["snapshot"]
pd.DataFrame(config["viewer_sequence"])

In [ ]:
pd.Series({
    "status": results["baseline_status"],
    "N": config["complexity_control_n"],
    "lambda": config["voting_threshold"],
    "seed": config["seed"],
    "configuration_sha256": results["configuration"]["sha256"],
}, name="value").rename_axis("setting").reset_index()

## Residual Cascade Training

In [ ]:
pd.DataFrame(results["cascade_training_trace"])

## Validation And Final-Test Metrics

In [ ]:
metric_names = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc"]
experimental = results["experimental_reconstruction"]
pd.DataFrame({
    split: {metric: experimental[split][metric] for metric in metric_names}
    for split in ("validation", "test")
}).rename_axis("metric").reset_index()

## Published Versus Independent Reconstruction

In [ ]:
comparison = pd.DataFrame.from_dict(results["test_differences_from_published"], orient="index")
comparison.index.name = "metric"
comparison.reset_index()

In [ ]:
ax = comparison[["published", "experimental"]].plot.bar(color=["#4C78A8", "#F58518"])
ax.set_title("Published MulCas vs independent final-test reconstruction")
ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()

## Final-Test Confusion Matrix

In [ ]:
test = experimental["test"]
pd.DataFrame(
    [[test["tn"], test["fp"]], [test["fn"], test["tp"]]],
    index=["Actual negative", "Actual positive"],
    columns=["Predicted negative", "Predicted positive"],
)

## Routing And Trace Inspection

In [ ]:
routing = pd.DataFrame({
    split: pd.Series(experimental[split]["routing_counts"])
    for split in ("validation", "test")
}).fillna(0).astype(int)
routing

In [ ]:
ax = routing.plot.bar()
ax.set_title("Samples resolved by cascade stage or vote")
ax.set_xlabel("Handler")
ax.set_ylabel("Contracts")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

In [ ]:
test_trace.sort_values(["sample_index", "stage", "phase"]).head(30)